# EM synapse mapping - multi-neuron use case

Creating a circuit from a set of morphologies with spines.

# Authentication

Follow the instructions below to authenticate and select a project to work in.

The morphology-with-spines instances that are to serve as the input of circuit generation must be accessible from the project.

In [ ]:
import obi_auth

from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities

from entitysdk.models import CellMorphology, EMCellMesh, EMDenseReconstructionDataset, Circuit
from entitysdk import Client
from entitysdk._server_schemas import AssetLabel

token = obi_auth.get_token(environment="production", auth_mode="daf")
project_context = get_projects(token, env="production")

# Find skeletonized morphologies as input

We search for morphologies originating from the `IARPA MICrONS mouse`, as they are derived by skeletonization

In [ ]:
entity_client = Client(token_manager=token, project_context=project_context, environment="production")

results = entity_client.search_entity(
    entity_type=CellMorphology,
    query={"subject__name": "IARPA MICrONS mouse"},
).all()

# Filter by morphology name, avoid duplicates
morph_names = []
spiny_morphologies = []
for morph in results:
    if morph.name not in morph_names:
        morph_names.append(morph.name)
        spiny_morphologies.append(morph)

print(f"Found {len(spiny_morphologies)} spiny morphologies:")
for m in spiny_morphologies:
    print(f"  - {m.name} (id={m.id})")


## Get `pt_root_id`. 
Next, we have to link the skeletons to their originating cell mesh. In the absence of an activity linking the two, we match them via the source identifier, the `pt_root_id`.

We print the description of the neurons below.

In [ ]:
from entitysdk.models import SkeletonizationExecution, EMCellMesh
from obi_one.scientific.from_id.cell_morphology_from_id import CellMorphologyFromID
from obi_one.scientific.tasks.em_synapse_mapping.resolve_neuron import resolve_provenance

# {pt_root_id: {"morph_from_id": CellMorphologyFromID, "name": str}}
neurons = {}

for morph_entity in spiny_morphologies:
    morph_from_id = CellMorphologyFromID(id_str=str(morph_entity.id))

    if not morph_from_id.has_source_mesh(db_client=entity_client):
        print(f"  WARNING: Could not resolve provenance for {morph_entity.name}, skipping.")
        continue

    pt_root_id, source_mesh, _ = resolve_provenance(entity_client, morph_from_id)
    if source_mesh.release_version > 1412:

        print(f"{morph_entity.description}")
        print(f"  {morph_entity.name} -> pt_root_id={pt_root_id}")
        neurons[pt_root_id] = {"morph_from_id": morph_from_id, "name": morph_entity.name}

print(f"\nTotal neuron entries: {len(neurons)}")

## Inspect connectivity between neurons

Query CAVE for synapses among the resolved neurons and show, for each physical neuron, which other neurons it connects to (degrees).

First, create a CAVE token

In [ ]:
import caveclient
temporary_client = caveclient.CAVEclient()
temporary_client.auth.get_new_token()

Paste your token below:

In [ ]:
import os
import caveclient
os.environ["CAVECLIENT_MICRONS_API_KEY"] = "PASTE_YOUR_TOKEN_HERE"

Inspect connectivity between the neurons in the set.

In [ ]:
# Ignore caveclient/materializationengine.py:1745 -- RuntimeWarning
import warnings
warnings.filterwarnings("ignore", message="Engine has switched to", category=RuntimeWarning)

CAVE_DATASTACK = "minnie65_public"

cave = caveclient.CAVEclient(CAVE_DATASTACK, auth_token=os.environ["CAVECLIENT_MICRONS_API_KEY"])
cave.version = 1507
pt_root_ids = list(neurons.keys())

assert cave.materialize.version == 1507

syns = cave.materialize.synapse_query(post_ids=pt_root_ids, pre_ids=pt_root_ids)
raw_mat = syns.groupby(["pre_pt_root_id", "post_pt_root_id"])["valid"].count().unstack("post_pt_root_id", fill_value=0)
mat = raw_mat.reindex(columns=pt_root_ids, index=pt_root_ids).fillna(0)
print(mat)

Now we pick the three neurons with the highest degrees.

In [ ]:
import numpy

#degrees
mat = mat.to_numpy()
idxx = numpy.argsort(mat.sum(axis=0) + mat.sum(axis=1))[-3:]
circuit_pt_root_ids = numpy.array(pt_root_ids)[idxx]

print(f"Neurons with highest degrees, in order:\n{circuit_pt_root_ids}")
print(f"Degrees:\n{mat[:, idxx][idxx]}")

In [ ]:
circuit_entities = [neurons[pt_id] for pt_id in circuit_pt_root_ids]
print(f"Circuit neurons with their name in the circuit:")
for neuron in circuit_entities:
    print(f"  {neuron["name"]}: {neuron["morph_from_id"].id_str}")

circuit_neurons = [entity["morph_from_id"] for entity in circuit_entities]

# Set up the multi-neuron synapse mapping task

In [ ]:
from obi_one.scientific.tasks.em_synapse_mapping.config import EMSynapseMappingSingleConfig, AdvancedEMSynapseMappingOptions
from obi_one.scientific.tasks.em_synapse_mapping.task import EMSynapseMappingTask
from obi_one.scientific.from_id.named_tuple_from_id import EMSynapseMappingInputNamedTuple
from obi_one.core.info import Info
from pathlib import Path

# We take the last 3 characters of the morphology UUID to easily identify the neurons
circuit_neurons_short_names = [n.id_str[-3:] for n in circuit_neurons]

OUTPUT_DIR = "multi-neuron-circuit-" + "-".join(circuit_neurons_short_names)
suffix = 1
while Path(OUTPUT_DIR).exists():
    OUTPUT_DIR = "multi-neuron-circuit-" + "-".join(circuit_neurons_short_names) + f"_{suffix}"
    suffix += 1

task_config = EMSynapseMappingSingleConfig(
    info=Info(
        campaign_name="EM Synapse Mapping Multi-Neuron",
        campaign_description="Map EM synapses onto multiple spiny morphologies"
    ),
    initialize=EMSynapseMappingSingleConfig.Initialize(
        neurons=EMSynapseMappingInputNamedTuple(name="Skeletons", elements=tuple(circuit_neurons)),
    ),
    coordinate_output_root=OUTPUT_DIR,
    advanced_options=AdvancedEMSynapseMappingOptions()
)

task = EMSynapseMappingTask(config=task_config)
print(f"Running synapse mapping task with {len(circuit_neurons)} neurons...")
print(f"Output: {OUTPUT_DIR}/")
task.execute(db_client=entity_client)

## Inspect the output (local files)

In [ ]:
import json
import h5py
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
config_path = output_dir / "circuit_config.json"

if config_path.exists():
    with open(config_path) as f:
        circuit_cfg = json.load(f)
    print(json.dumps(circuit_cfg, indent=2))

    edges_file = output_dir / "synaptome-edges.h5"
    if edges_file.exists():
        with h5py.File(edges_file, "r") as h5:
            print("\nEdge populations:")
            for pop_name in h5["edges"]:
                grp = h5[f"edges/{pop_name}"]
                n = len(grp["source_node_id"])
                src_pop = grp["source_node_id"].attrs.get("node_population", "?")
                tgt_pop = grp["target_node_id"].attrs.get("node_population", "?")
                print(f"  {pop_name}: {n} edges ({src_pop} -> {tgt_pop})")
else:
    print("No circuit output found. Run the task cell above first.")

## Test circuit validity with SNAP

Display the biophysical node population properties (one row per neuron)


In [ ]:
import bluepysnap as snap

circ = snap.Circuit(OUTPUT_DIR + "/circuit_config.json")
bio_pop = circ.nodes["biophysical_neurons"]
bio_pop.get()

Verify that SWC and HDF5 morphology files referenced by the circuit can be loaded

In [ ]:
print(f"SWC: {bio_pop.morph.get(0, extension="swc")}")
print(f"HDF5: {bio_pop.morph.get(0, extension="h5")}")

Load the first morphology of the circuit as a spiny morphology

In [ ]:
from morph_spines import load_morphology_with_spines

spiny_morph = load_morphology_with_spines(bio_pop.config["alternate_morphologies"]["h5v1"],
                            bio_pop.get(0, properties="morphology").split("/")[1])

print(spiny_morph)